In [ ]:
%pip install jsonschema
%pip install langchain
%pip install langchain_mistralai
%pip install pandas
%pip install aijsondbpy

We will create an **agent** that can **answer questions based on data** found in a **JSON file**.
**LangChain** will be used as the **agent framework**.

For the agent, we need:
- A **system prompt**
- The **tool(s)** the agent will use

Let’s start with the **tool**.

To query the JSON data with **pure JavaScript**, we use the **aijsondb** database, which allows querying JSON data using **pure JavaScript**. The **aijsondb** database is initialized with the **sample JSON data** and the **JSON schema** located in the **data folder**.

In [13]:
import aijsondb
aijsondb.init_db("./data/500 KB_V3.json","./data/employeeSchemaDescription_V3.json")

Now we can query the database. It contains **201 employee records**. The number of employees can be **obtained** by:
```
var result=data.employees.length
```


In [15]:
aijsondb.query_data_javascript("var result=data.employees.length")

201


The JavaScript query engine is working. The next step is to create a function that can be used for tool calls.

In [16]:
from langchain.tools import tool
last_answer_from_tool=None
@tool
def query_json_javascript(query: str) -> any:
    """Run a javascript expression on the provided data."""
    global last_answer_from_tool
    try:
        last_answer_from_tool=matches=None
        matches=aijsondb.query_data_javascript(query)
        last_answer_from_tool=matches
        return matches
    except Exception as e:  
        serr=f"Error running JSONPath query: {e}"
        return serr 

Next, we create the **system prompt** to guide the agent. This prompt will include:
- Instructions on how to use the tool
- A **schema description** of the data, so the agent can understand the structure of the data it is working with.

In [19]:
import json
with open('../data/talktodata/employeeSchemaDescription_V3.json') as f:
    jschema = json.load(f)  

schema = json.dumps(jschema, indent=4)

template =f"""
You are a JavaScript query assistant for a JSON data object. Your task is to generate a JavaScript expression to fetch data from the object based on a natural language question.

### JSON Schema:
The data follows this schema:
{schema}

### Rules:
1. **Data Access**:
   - The JSON object is stored in a variable named `data`.
   - The query result must be assigned to a variable named `result` (e.g., `var result = ...`).

2. **Query Generation**:
   - Only generate a JavaScript expression if the question can be answered using the provided schema.
   - If the question **cannot** be answered (e.g., the data does not exist in the schema or the question is irrelevant), respond with:
     *"This question cannot be answered with the available data."*

3. **Tool Usage**:
   - If the question is valid, use the tool:
     `query_json_javascript` (returns data using the generated JavaScript query).
   - Do **not** call the tool if the question is unanswerable.

"""

Now that we have the **tool function** and the **system prompt** for the agent, we can create the agent.

We will use the **Mistral AI model**: `mistral-large-latest` for the agent. Please note that I have saved the **API key** from Mistral in the environment variable `MISTRAL_API_KEY`.

In [22]:
from langchain_mistralai.chat_models import ChatMistralAI
from langchain.agents import create_agent

llmm = ChatMistralAI(
    model="mistral-large-latest",
    temperature=0,
    max_retries=3,
)

agent = create_agent(
    model=llmm,
    tools=[query_json_javascript],
    system_prompt=template,
)

That’s it! Now we can ask the agent questions about the data, and it will generate the appropriate **JavaScript queries** to fetch the answers. If a question cannot be answered with the available data, the agent will respond accordingly.

In [30]:
from langchain.messages import HumanMessage 
result = agent.invoke(
    {"messages": [HumanMessage("Hown many employees are there?")]}
)
result['messages'][-1].content

'There are **201 employees** in the data.'

The result is correct—there are **201 employees** in the dataset.
The query generated by the LLM is:

In [33]:
result['messages'][1].additional_kwargs["tool_calls"][0]

{'id': 'NAOpVLIO1',
 'function': {'name': 'query_json_javascript',
  'arguments': '{"query": "var result = data.employees.length;"}'},
 'index': 0}

The query created by the LLM is identical to the one we used earlier.
Now, let’s see how it performs with a more complex query.

In [35]:
from langchain.messages import HumanMessage 
result = agent.invoke(
    {"messages": [HumanMessage("Which employees have the greatest experience?")]}
)
print(result['messages'][-1].content)

The following employees have the greatest experience (10 years):

1. Eileen Fitzgerald
2. Ashley Arroyo
3. Blake Mitchell
4. Stephanie Lewis
5. Chris Thomas
6. Brenda Wang
7. David Smith
8. Patrick Lee
9. Jim Werner
10. Abigail Briggs
11. Cathy O'Connor
12. Ricky Perry
13. Billy Bradley DVM
14. Matthew Caldwell
15. Robert Kelley
16. Alexander Guerrero IV
17. Rebecca King
18. Barbara Andrews
19. Dustin Newton
20. Sara Rivera
21. Duane Aguilar
22. April Cline
23. Robert Johnson
24. Willie Jones
25. Kelly Cardenas
26. Jennifer Anderson
27. Mary Fox
28. Albert Gonzalez
29. Timothy Mullins


The result is correct: **all employees have 10 years of experience**.
The query is now **much more complex**.

In [43]:
args=json.loads(result['messages'][1].additional_kwargs["tool_calls"][0]['function']['arguments'])
print(args['query'])

// Get all employees and map their experience years
var result = data.employees
    .filter(emp => emp.profile.projects && emp.profile.projects.some(project => 
        project.tasks && project.tasks.some(task => 
            task.assignedTo && task.assignedTo.skills && task.assignedTo.skills.experience && 
            task.assignedTo.skills.experience.years !== undefined
        )
    ))
    .map(emp => {
        // Find the maximum experience years across all tasks and projects
        let maxExperience = 0;
        emp.profile.projects.forEach(project => {
            project.tasks.forEach(task => {
                if (task.assignedTo && task.assignedTo.skills && task.assignedTo.skills.experience) {
                    const years = task.assignedTo.skills.experience.years;
                    if (years > maxExperience) {
                        maxExperience = years;
                    }
                }
            });
        });
        return {
            id: emp.id,
        